<a href="https://colab.research.google.com/github/sandyreda898-oss/Smart-Agricultural-Water-Management/blob/main/weather_soil_water_modeling/notebooks/05_etc_calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

repo_path = "/content/Smart-Agricultural-Water-Management"

if not os.path.exists(repo_path):
    !git clone https://github.com/sandyreda898-oss/Smart-Agricultural-Water-Management.git
else:
    print("Repository already exists. Using the current copy.")

Cloning into 'Smart-Agricultural-Water-Management'...
remote: Enumerating objects: 709, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 709 (delta 49), reused 0 (delta 0), pack-reused 562 (from 1)
Receiving objects: 100% (709/709), 157.67 MiB | 17.27 MiB/s, done.
Resolving deltas: 100% (288/288), done.
Updating files: 100% (449/449), done.


In [2]:
import pandas as pd
import numpy as np

eto_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/eto_daily.csv.gz"
)

stage_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/reference/rice_growth_stage_kc_reference.csv"
)

eto = pd.read_csv(
    eto_path,
    parse_dates=["obs_date"]
)

stage_reference = pd.read_csv(stage_path)

print("ET0 shape:", eto.shape)
print("Stage reference shape:", stage_reference.shape)

ET0 shape: (136002, 39)
Stage reference shape: (4, 8)


In [4]:
eto.head()

,point_id,obs_date,ALLSKY_SFC_SW_DWN,PRECTOTCORR,PS,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M,...,Ra,Rs,Rso,rs_rso_raw,radiation_ratio_review_flag,rs_rso,Rns,Rnl,Rn,ET0_mm_day
0,grid_0001,2017-01-01,11.32,0.04,102.26,64.90,9.79,16.73,5.38,2.26,...,19.371021,11.32,14.528266,0.779171,False,0.779171,8.7164,4.648914,4.067486,1.862139
1,grid_0001,2017-01-02,10.49,0.22,102.33,73.94,10.76,16.53,5.64,2.28,...,19.423812,10.49,14.567859,0.720078,False,0.720078,8.0773,3.945966,4.131334,1.566919
2,grid_0001,2017-01-03,11.33,0.15,102.42,74.23,11.64,17.38,6.94,1.60,...,19.480760,11.33,14.610570,0.775466,False,0.775466,8.7241,4.372217,4.351883,1.498149
3,grid_0001,2017-01-04,10.52,0.39,102.29,80.24,12.06,17.60,7.99,2.21,...,19.541843,10.52,14.656382,0.717776,False,0.717776,8.1004,3.745384,4.355016,1.460555
4,grid_0001,2017-01-05,10.12,0.00,102.14,63.24,11.77,17.80,5.71,1.33,...,19.607038,10.12,14.705278,0.688188,False,0.688188,7.7924,3.839618,3.952782,1.591531


In [5]:
stage_reference

,growth_stage,stage_length_days,stage_length_unit,stage_length_status,kc,kc_min,kc_max,kc_status
0,initial,30.0,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED,1.05,1.05,1.05,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
1,development,30.0,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED,NaN,NaN,NaN,NaN
2,mid_season,60.0,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED,1.20,1.20,1.20,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
3,late_season,30.0,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED,NaN,0.60,0.90,FAO_REFERENCE_RANGE_NOT_LOCALLY_CALIBRATED


In [6]:
print("ET0 columns:")
print(eto.columns.tolist())

print("\nStage reference columns:")
print(stage_reference.columns.tolist())

ET0 columns:
['point_id', 'obs_date', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'PS', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'latitude', 'longitude', 'district_id', 'spatial_support', 'et0_inputs_complete', 'rainfall_available', 'rainfall_review_flag', 'day_of_year', 'latitude_rad', 'es_tmax', 'es_tmin', 'es', 'ea', 'vpd', 'delta', 'gamma', 'inverse_relative_distance', 'solar_declination', 'sunset_hour_angle', 'Ra', 'Rs', 'Rso', 'rs_rso_raw', 'radiation_ratio_review_flag', 'rs_rso', 'Rns', 'Rnl', 'Rn', 'ET0_mm_day']

Stage reference columns:
['growth_stage', 'stage_length_days', 'stage_length_unit', 'stage_length_status', 'kc', 'kc_min', 'kc_max', 'kc_status']


In [7]:
print("ET0 missing values:")
print(
    eto[
        ["point_id", "obs_date", "ET0_mm_day"]
    ].isnull().sum()
)

print("\nStage reference missing values:")
print(stage_reference.isnull().sum())

ET0 missing values:
point_id      0
obs_date      0
ET0_mm_day    0
dtype: int64

Stage reference missing values:
growth_stage           0
stage_length_days      0
stage_length_unit      0
stage_length_status    0
kc                     2
kc_min                 1
kc_max                 1
kc_status              1
dtype: int64


In [8]:
season_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/reference/season_reference.csv"
)

season = pd.read_csv(
    season_path,
    parse_dates=["start_date", "end_date"]
)

season

,season_year,start_date,end_date
0,2017,2017-05-10,2017-10-31
1,2018,2018-05-10,2018-10-31
2,2019,2019-05-10,2019-10-31
3,2020,2020-05-10,2020-10-31
4,2021,2021-05-10,2021-10-31
5,2022,2022-05-10,2022-10-31
6,2023,2023-05-10,2023-10-31


In [9]:
etc = eto.copy()

etc["season_year"] = etc["obs_date"].dt.year

etc = etc.merge(
    season,
    on="season_year",
    how="inner",
    validate="many_to_one"
)

print("Rows after limiting to documented seasons:", len(etc))

Rows after limiting to documented seasons: 97128


In [11]:
# Calculate day within the rice season
etc["day_in_season"] = (
    etc["obs_date"] -
    etc["start_date"]
).dt.days + 1

In [12]:
etc = etc[
    (etc["day_in_season"] >= 1) &
    (etc["day_in_season"] <= 150)
].copy()

print("Rice-season rows:", len(etc))

Rice-season rows: 39900


In [13]:
# Assign growth stages
conditions = [
    etc["day_in_season"].between(1, 30),
    etc["day_in_season"].between(31, 60),
    etc["day_in_season"].between(61, 120),
    etc["day_in_season"].between(121, 150)
]

stages = [
    "initial",
    "development",
    "mid_season",
    "late_season"
]

etc["growth_stage"] = np.select(
    conditions,
    stages,
    default="unknown"
)

etc["growth_stage"].value_counts()

,count
growth_stage,
mid_season,15960
initial,7980
development,7980
late_season,7980


In [14]:
# Calculate daily Kc
etc["kc_low"] = np.nan
etc["kc_high"] = np.nan

In [15]:
initial_mask = etc["growth_stage"] == "initial"

etc.loc[initial_mask, "kc_low"] = 1.05
etc.loc[initial_mask, "kc_high"] = 1.05

In [16]:
development_mask = (
    etc["growth_stage"] == "development"
)

development_fraction = (
    etc.loc[development_mask, "day_in_season"] - 30
) / 30

etc.loc[development_mask, "kc_low"] = (
    1.05 +
    development_fraction *
    (1.20 - 1.05)
)

etc.loc[development_mask, "kc_high"] = (
    etc.loc[development_mask, "kc_low"]
)

In [17]:
mid_mask = etc["growth_stage"] == "mid_season"

etc.loc[mid_mask, "kc_low"] = 1.20
etc.loc[mid_mask, "kc_high"] = 1.20

In [18]:
late_mask = etc["growth_stage"] == "late_season"

late_fraction = (
    etc.loc[late_mask, "day_in_season"] - 120
) / 30

etc.loc[late_mask, "kc_low"] = (
    1.20 +
    late_fraction *
    (0.60 - 1.20)
)

etc.loc[late_mask, "kc_high"] = (
    1.20 +
    late_fraction *
    (0.90 - 1.20)
)

In [19]:
# Inspect Kc
etc[
    [
        "day_in_season",
        "growth_stage",
        "kc_low",
        "kc_high"
    ]
].drop_duplicates().sort_values(
    "day_in_season"
).head(160)

,day_in_season,growth_stage,kc_low,kc_high
129,1,initial,1.05,1.05
130,2,initial,1.05,1.05
131,3,initial,1.05,1.05
132,4,initial,1.05,1.05
133,5,initial,1.05,1.05
...,...,...,...,...
274,146,late_season,0.68,0.94
275,147,late_season,0.66,0.93
276,148,late_season,0.64,0.92
277,149,late_season,0.62,0.91


In [20]:
# Validate Kc
print("Missing Kc low:", etc["kc_low"].isna().sum())
print("Missing Kc high:", etc["kc_high"].isna().sum())

print("Kc low range:")
print(etc["kc_low"].describe())

print("\nKc high range:")
print(etc["kc_high"].describe())

Missing Kc low: 0
Missing Kc high: 0
Kc low range:
count    39900.000000
mean         1.093500
std          0.140703
min          0.600000
25%          1.050000
50%          1.140000
75%          1.200000
max          1.200000
Name: kc_low, dtype: float64

Kc high range:
count    39900.000000
mean         1.124500
std          0.080803
min          0.900000
25%          1.050000
50%          1.150000
75%          1.200000
max          1.200000
Name: kc_high, dtype: float64


In [21]:
# Calculate ETc
etc["ETc_low_mm_day"] = (
    etc["ET0_mm_day"] *
    etc["kc_low"]
)

etc["ETc_high_mm_day"] = (
    etc["ET0_mm_day"] *
    etc["kc_high"]
)

In [22]:
# Validate ETc
print(
    "Missing ETc low:",
    etc["ETc_low_mm_day"].isna().sum()
)

print(
    "Missing ETc high:",
    etc["ETc_high_mm_day"].isna().sum()
)

print(
    "Negative ETc low:",
    (etc["ETc_low_mm_day"] < 0).sum()
)

print(
    "Negative ETc high:",
    (etc["ETc_high_mm_day"] < 0).sum()
)

Missing ETc low: 0
Missing ETc high: 0
Negative ETc low: 0
Negative ETc high: 0


In [24]:
# summary
etc[
    [
        "ET0_mm_day",
        "kc_low",
        "kc_high",
        "ETc_low_mm_day",
        "ETc_high_mm_day"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
ET0_mm_day,39900.0,6.735555,1.001994,3.351625,6.075874,6.687013,7.372880,11.627580
kc_low,39900.0,1.093500,0.140703,0.600000,1.050000,1.140000,1.200000,1.200000
kc_high,39900.0,1.124500,0.080803,0.900000,1.050000,1.150000,1.200000,1.200000
ETc_low_mm_day,39900.0,7.437855,1.672500,2.397562,6.561977,7.714991,8.554694,12.208959
ETc_high_mm_day,39900.0,7.607373,1.395876,3.446495,6.697296,7.727368,8.556080,12.208959


In [25]:
# Seasonal totals
seasonal_etc = (
    etc.groupby(
        ["point_id", "season_year"]
    )
    .agg(
        ETc_low_season_mm=(
            "ETc_low_mm_day",
            "sum"
        ),
        ETc_high_season_mm=(
            "ETc_high_mm_day",
            "sum"
        )
    )
    .reset_index()
)

seasonal_etc.head()

,point_id,season_year,ETc_low_season_mm,ETc_high_season_mm
0,grid_0001,2017,1202.974001,1228.196566
1,grid_0001,2018,1203.716044,1230.534286
2,grid_0001,2019,1224.860108,1250.426308
3,grid_0001,2020,1197.470565,1224.723246
4,grid_0001,2021,1231.234540,1258.247982


In [26]:
seasonal_etc[
    [
        "ETc_low_season_mm",
        "ETc_high_season_mm"
    ]
].describe()

,ETc_low_season_mm,ETc_high_season_mm
count,266.000000,266.000000
mean,1115.678323,1141.105919
std,83.550181,84.416859
min,1012.330225,1036.592319
25%,1041.657613,1065.076656
50%,1063.440367,1089.242599
75%,1203.676894,1230.486909
max,1234.911432,1261.387850


In [27]:
etc_output_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/etc_daily.csv.gz"
)

etc.to_csv(
    etc_output_path,
    index=False,
    compression="gzip"
)

print("Daily ETc saved.")
print(etc_output_path)

Daily ETc saved.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/etc_daily.csv.gz


In [28]:
seasonal_output_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/etc_seasonal.csv"
)

seasonal_etc.to_csv(
    seasonal_output_path,
    index=False
)

print("Seasonal ETc saved.")
print(seasonal_output_path)

Seasonal ETc saved.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/etc_seasonal.csv


In [29]:
import os

print(
    "Daily exists:",
    os.path.exists(etc_output_path)
)

print(
    "Seasonal exists:",
    os.path.exists(seasonal_output_path)
)

print("Daily rows:", len(etc))
print("Seasonal rows:", len(seasonal_etc))

Daily exists: True
Seasonal exists: True
Daily rows: 39900
Seasonal rows: 266


In [30]:
from google.colab import files

files.download(etc_output_path)
files.download(seasonal_output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>